# JetGuard - LSTM Sequence Model


Train an LSTM on engine-bounded FD001 sensor sequences. Window length is selected using validation engines only. The test set is evaluated once using the selected validation model.


Features are the three operating settings and fifteen non-constant sensors. The scaler is fitted only on observations from training engines.

In [1]:
from pathlib import Path
import copy
import json
import random
import time

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATASET_DIR = PROJECT_ROOT / "dataset"
REPORT_DIR = PROJECT_ROOT / "reports"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
WINDOW_LENGTHS = [20, 30, 50, 60]
HIDDEN_SIZE = 64
BATCH_SIZE = 256
MAX_EPOCHS = 80
PATIENCE = 8
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cpu


In [2]:
with (PROCESSED_DIR / "feature_metadata.json").open("r", encoding="utf-8") as file:
    feature_metadata = json.load(file)

train_df = pd.read_csv(PROCESSED_DIR / "train_FD001_processed.csv")
split_train_engines = set(feature_metadata["training_engines"])
split_validation_engines = set(feature_metadata["validation_engines"])
sensor_columns = feature_metadata["informative_sensors"]
feature_columns = ["setting_1", "setting_2", "setting_3"] + sensor_columns

test_columns = ["engine_id", "cycle"] + [f"setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
test_df = pd.read_csv(DATASET_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=test_columns)
test_rul = pd.read_csv(DATASET_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_true"])["RUL_true"].to_numpy()

train_split = train_df[train_df.engine_id.isin(split_train_engines)].copy()
validation_split = train_df[train_df.engine_id.isin(split_validation_engines)].copy()

assert split_train_engines.isdisjoint(split_validation_engines)
assert len(test_rul) == test_df.engine_id.nunique()
print("Train engines:", train_split.engine_id.nunique())
print("Validation engines:", validation_split.engine_id.nunique())
print("Test engines:", test_df.engine_id.nunique())
print("Input features:", len(feature_columns))

Train engines: 80
Validation engines: 20
Test engines: 100
Input features: 18


In [3]:
scaler = StandardScaler()
scaler.fit(train_split[feature_columns])

def scale_frame(frame):
    result = frame.sort_values(["engine_id", "cycle"]).copy()
    scaled_values = scaler.transform(result[feature_columns])
    for index, column in enumerate(feature_columns):
        result[column] = scaled_values[:, index]
    return result

train_scaled = scale_frame(train_split)
validation_scaled = scale_frame(validation_split)
test_scaled = scale_frame(test_df)
joblib.dump(scaler, ARTIFACT_DIR / "lstm_standard_scaler.joblib")
print("Scaler fit on training-engine observations only.")

Scaler fit on training-engine observations only.


In [4]:
def make_training_windows(frame, window_length):
    sequences, targets = [], []
    for _, engine in frame.groupby("engine_id", sort=True):
        engine = engine.sort_values("cycle")
        values = engine[feature_columns].to_numpy(dtype=np.float32)
        rul = engine["RUL_capped"].to_numpy(dtype=np.float32)
        for end in range(window_length - 1, len(engine)):
            start = end - window_length + 1
            sequences.append(values[start:end + 1])
            targets.append(rul[end])
    return np.stack(sequences), np.asarray(targets, dtype=np.float32)


def make_test_windows(frame, window_length, true_rul):
    sequences, engine_ids, final_cycles = [], [], []
    for engine_id, engine in frame.groupby("engine_id", sort=True):
        engine = engine.sort_values("cycle")
        values = engine[feature_columns].to_numpy(dtype=np.float32)
        if len(values) >= window_length:
            values = values[-window_length:]
        else:
            pad = np.repeat(values[:1], window_length - len(values), axis=0)
            values = np.concatenate([pad, values], axis=0)
        sequences.append(values)
        engine_ids.append(int(engine_id))
        final_cycles.append(int(engine["cycle"].iloc[-1]))
    order = np.argsort(engine_ids)
    target_by_engine = dict(zip(sorted(frame.engine_id.unique()), true_rul))
    targets = np.asarray([target_by_engine[engine_ids[index]] for index in order], dtype=np.float32)
    return np.stack(sequences)[order], np.asarray(engine_ids)[order], np.asarray(final_cycles)[order], targets


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class RULLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, sequence):
        outputs, _ = self.lstm(sequence)
        return self.output(outputs[:, -1, :]).squeeze(1)

In [5]:
def nasa_rul_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalties = np.where(error < 0, np.exp(-error / 13.0) - 1.0, np.exp(error / 10.0) - 1.0)
    return float(np.sum(penalties))


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.clip(np.asarray(y_pred), 0.0, 125.0)
    nonzero = np.abs(y_true) > 1e-8
    mape = float(mean_absolute_percentage_error(y_true[nonzero], y_pred[nonzero])) if np.any(nonzero) else float("nan")
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": mape,
        "NASA_score": nasa_rul_score(y_true, y_pred),
    }, y_pred


def train_for_window(window_length):
    seed_everything(RANDOM_SEED)
    x_train, y_train = make_training_windows(train_scaled, window_length)
    x_validation, y_validation = make_training_windows(validation_scaled, window_length)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    validation_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_validation), torch.from_numpy(y_validation)),
        batch_size=BATCH_SIZE, shuffle=False,
    )

    model = RULLSTM(len(feature_columns), HIDDEN_SIZE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_function = nn.MSELoss()
    best_state = None
    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            predictions = model(batch_x)
            loss = loss_function(predictions, batch_y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        validation_losses = []
        with torch.no_grad():
            for batch_x, batch_y in validation_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                validation_losses.append(loss_function(model(batch_x), batch_y).item())
        train_loss = float(np.mean(train_losses))
        validation_loss = float(np.mean(validation_losses))
        history.append({"epoch": epoch, "train_mse": train_loss, "validation_mse": validation_loss})
        print(f"window={window_length} epoch={epoch} train_mse={train_loss:.3f} validation_mse={validation_loss:.3f}")

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break

    if best_state is None:
        raise RuntimeError(f"No model checkpoint was produced for window {window_length}.")
    model.load_state_dict(best_state)
    return model, best_epoch, best_validation_loss, history

In [6]:
window_results = []
training_histories = {}
trained_models = {}

for window_length in WINDOW_LENGTHS:
    started = time.perf_counter()
    model, best_epoch, best_loss, history = train_for_window(window_length)
    elapsed = time.perf_counter() - started
    trained_models[window_length] = model
    training_histories[window_length] = history

    x_validation, y_validation = make_training_windows(validation_scaled, window_length)
    model.eval()
    with torch.no_grad():
        validation_predictions = model(torch.from_numpy(x_validation).to(DEVICE)).cpu().numpy()
    validation_metrics, validation_predictions = calculate_metrics(y_validation, validation_predictions)
    result = {
        "window_length": window_length,
        "best_epoch": best_epoch,
        "best_validation_mse": best_loss,
        "training_seconds": elapsed,
        **validation_metrics,
    }
    window_results.append(result)
    torch.save(model.state_dict(), ARTIFACT_DIR / f"lstm_window_{window_length}.pth")

validation_results = pd.DataFrame(window_results).sort_values("NASA_score").reset_index(drop=True)
validation_results.to_csv(REPORT_DIR / "lstm_validation_window_comparison.csv", index=False)
validation_results

window=20 epoch=1 train_mse=8305.104 validation_mse=6969.333


window=20 epoch=2 train_mse=7239.522 validation_mse=6257.471


window=20 epoch=3 train_mse=6573.440 validation_mse=5691.322


window=20 epoch=4 train_mse=6009.640 validation_mse=5198.664


window=20 epoch=5 train_mse=5516.025 validation_mse=4758.114


window=20 epoch=6 train_mse=5064.173 validation_mse=4359.417


... (258 outputs left out)

window=60 epoch=72 train_mse=47.069 validation_mse=197.104


window=60 epoch=73 train_mse=44.643 validation_mse=180.158


window=60 epoch=74 train_mse=42.279 validation_mse=189.127


,window_length,best_epoch,best_validation_mse,training_seconds,RMSE,MAE,MAPE,NASA_score
0,50,55,151.115612,57.446159,12.673987,9.813985,0.167465,6240.983398
1,60,66,162.151029,76.291907,12.793677,9.192779,0.158750,6615.957031
2,30,59,214.042204,45.732708,14.680404,10.695864,0.175076,20965.382812
3,20,55,283.407976,36.681722,16.845461,12.387916,0.207948,30374.976562


In [7]:
selected_window = int(validation_results.iloc[0]["window_length"])
selected_model = trained_models[selected_window]
x_test, test_engine_ids, test_final_cycles, y_test = make_test_windows(test_scaled, selected_window, test_rul)

selected_model.eval()
with torch.no_grad():
    test_predictions = selected_model(torch.from_numpy(x_test).to(DEVICE)).cpu().numpy()
test_metrics, test_predictions = calculate_metrics(y_test, test_predictions)

test_output = pd.DataFrame({
    "engine_id": test_engine_ids,
    "final_cycle": test_final_cycles,
    "true_RUL": y_test,
    "predicted_RUL": test_predictions,
})
test_output.to_csv(REPORT_DIR / "lstm_test_predictions.csv", index=False)
pd.DataFrame([{"model": "LSTM", "window_length": selected_window, **test_metrics}]).to_csv(
    REPORT_DIR / "lstm_test_metrics.csv", index=False
)
torch.save(selected_model.state_dict(), ARTIFACT_DIR / "lstm_selected.pth")

run_metadata = {
    "seed": RANDOM_SEED,
    "device": str(DEVICE),
    "input_features": feature_columns,
    "window_lengths_tested": WINDOW_LENGTHS,
    "selected_window_length": selected_window,
    "selection_metric": "validation NASA_score (lower is better)",
    "hidden_size": HIDDEN_SIZE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "target": "RUL_capped for train/validation; official RUL for test",
    "test_metrics": test_metrics,
}
(ARTIFACT_DIR / "lstm_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Selected window:", selected_window)
print("Final test metrics:", json.dumps(test_metrics, indent=2))
test_output.head()

Selected window: 50
Final test metrics: {
  "RMSE": 16.860205826956673,
  "MAE": 12.675820350646973,
  "MAPE": 0.1807665228843689,
  "NASA_score": 407.59576416015625
}


,engine_id,final_cycle,true_RUL,predicted_RUL
0,1,31,112.0,107.315567
1,2,49,98.0,106.207825
2,3,126,69.0,59.541710
3,4,106,82.0,74.623276
4,5,98,91.0,74.844330
